# Notebook 70 — Structured Interaction Basis and Block-Conditioned Symbolic Field

Notebook 69 showed:

> Full orthogonalization improves coordinate independence but damages field reconstruction.

Notebook 70 tests the next refinement:

> Can interaction-aware, partially decoupled bases preserve raw-basis fit while improving stability and universality?

Compared basis classes:

1. `raw_6`: \(1,c,r,c^3,r^2,rc^2\)
2. `structured_interaction`: \(1,c,r,rc,c^3-\alpha c,r^2-\beta,rc^2\)
3. `compact_interaction`: \(1,c,r,rc,c^3,r^2\)
4. `block_conditioned`: compact for linear flow, nonlinear-interaction for nonlinear flow

Outputs:
- hard-block RMSE
- universality score
- stability / complexity comparison
- block-conditioned behavior
- final basis-class verdict

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, glob, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneOut

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

np.random.seed(42)

OUTPUT_DIR = "paper70_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
# Data loading and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet", "*residual*flow*.csv",
        "*governing*flow*.parquet", "*governing*flow*.csv",
        "*.parquet", "*.csv", "*.pkl", "*.pickle",
    ]
    for pat in patterns:
        candidates += glob.glob(pat)
        candidates += glob.glob(os.path.join("data", pat))
        candidates += glob.glob(os.path.join("outputs", pat))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        c_grid = np.linspace(-1.25, 1.05, 42)
                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(14):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                dc = c_grid[min(window_id + 1, len(c_grid)-1)] - c if window_id < len(c_grid)-1 else c - c_grid[max(window_id-1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * dc
                                rows.append({
                                    "system": system, "task": task, "forcing_mode": forcing_mode,
                                    "k": k, "flow_mode": flow_mode,
                                    "condition_coord": c, "residual": r,
                                    "predicted_flow": g + noise, "next_residual": next_r,
                                    "delta_condition": dc, "sample_id": sample_id,
                                    "path_id": path_id, "window_id": window_id,
                                })
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

def align_schema(df):
    alias_groups = {
        "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
        "residual": ["residual", "r", "resid"],
        "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
        "next_residual": ["next_residual", "r_next", "next_r"],
        "delta_condition": ["delta_condition", "dc", "delta_c"],
        "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
    }
    rename = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in df.columns and a != canonical:
                rename[a] = canonical
                break
    df = df.rename(columns=rename)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]
    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        df["delta_condition"] = np.gradient(df["condition_coord"].to_numpy())

    defaults = {
        "system": "unknown_system", "task": "unknown_task",
        "forcing_mode": "unknown_forcing", "k": 5,
        "flow_mode": "unknown_flow", "sample_id": np.arange(len(df)),
        "path_id": 0, "window_id": np.arange(len(df)),
    }
    for key, val in defaults.items():
        if key not in df.columns:
            df[key] = val

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Synthetic fallback:", USE_SYNTHETIC_FALLBACK)
print("Rows:", len(df))
display(df.head())

In [ ]:
# Basis definitions

def basis_stats(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return {
        "alpha_c3_on_c": float(np.sum(c**4) / max(np.sum(c**2), 1e-12)),
        "mean_r2": float(np.mean(r**2)),
    }

def raw_terms(data, stats=None):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return {
        "1": np.ones_like(r),
        "c": c,
        "r": r,
        "c^3": c**3,
        "r^2": r**2,
        "r c^2": r * c**2,
    }

def structured_terms(data, stats=None):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    if stats is None:
        stats = basis_stats(data)
    alpha = stats.get("alpha_c3_on_c", 0.0)
    beta = stats.get("mean_r2", 0.0)
    return {
        "1": np.ones_like(r),
        "c": c,
        "r": r,
        "r c": r * c,
        "c^3_perp_c": c**3 - alpha * c,
        "r^2_centered": r**2 - beta,
        "r c^2": r * c**2,
    }

def compact_interaction_terms(data, stats=None):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return {
        "1": np.ones_like(r),
        "c": c,
        "r": r,
        "r c": r * c,
        "c^3": c**3,
        "r^2": r**2,
    }

def nonlinear_interaction_terms(data, stats=None):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    if stats is None:
        stats = basis_stats(data)
    alpha = stats.get("alpha_c3_on_c", 0.0)
    beta = stats.get("mean_r2", 0.0)
    return {
        "1": np.ones_like(r),
        "c": c,
        "r": r,
        "r c": r * c,
        "c^3_perp_c": c**3 - alpha * c,
        "r^2_centered": r**2 - beta,
        "r c^2": r * c**2,
        "r^3": r**3,
    }

def get_basis_dict(data, basis_name, stats=None, flow_mode=None):
    if basis_name == "raw_6":
        return raw_terms(data, stats)
    if basis_name == "structured_interaction":
        return structured_terms(data, stats)
    if basis_name == "compact_interaction":
        return compact_interaction_terms(data, stats)
    if basis_name == "nonlinear_interaction":
        return nonlinear_interaction_terms(data, stats)
    if basis_name == "block_conditioned":
        if str(flow_mode) == "nonlinear":
            return nonlinear_interaction_terms(data, stats)
        return compact_interaction_terms(data, stats)
    raise ValueError(f"Unknown basis: {basis_name}")

def design_matrix(data, basis_name, stats=None, flow_mode=None, columns=None):
    d = get_basis_dict(data, basis_name, stats=stats, flow_mode=flow_mode)
    if columns is None:
        columns = list(d.keys())
    X = np.column_stack([d.get(col, np.zeros(len(data))) for col in columns])
    return X, columns

def fit_template(sub, basis_name, alpha=1e-6, flow_mode=None):
    stats = basis_stats(sub)
    X, names = design_matrix(sub, basis_name, stats=stats, flow_mode=flow_mode)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    row = {
        "n": len(sub),
        "template_rmse": math.sqrt(mean_squared_error(y, pred)),
        "basis_terms": "|".join(names),
    }
    for name, coef in zip(names, beta):
        row[f"coef_{name}"] = float(coef)
    return beta, pred, row, stats, names

def predict_with_beta(sub, basis_name, beta, stats=None, flow_mode=None, columns=None):
    X, names = design_matrix(sub, basis_name, stats=stats, flow_mode=flow_mode, columns=columns)
    return X @ np.asarray(beta, dtype=float)

def trajectory_gap(sub, basis_name, beta_ref, beta_cmp, stats_ref=None, stats_cmp=None, flow_mode=None, columns=None, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, stats, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i + 1] - cgrid[i])
            row = pd.DataFrame({"residual": [r], "condition_coord": [c]})
            g = float(np.clip(predict_with_beta(row, basis_name, beta, stats=stats, flow_mode=flow_mode, columns=columns)[0], -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    return float(np.mean([
        math.sqrt(mean_squared_error(integrate(beta_ref, stats_ref, r0), integrate(beta_cmp, stats_cmp, r0)))
        for r0 in r0s
    ]))

In [ ]:
# Coefficient table and symbolic chart helpers

def build_coef_table(basis_name):
    group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
    rows, subsets, stats_map, names_map = [], {}, {}, {}

    for vals, sub in df.groupby(group_cols):
        if len(sub) < 30:
            continue

        flow_mode = vals[4]
        beta, pred, stats_row, stats, names = fit_template(sub.copy(), basis_name, flow_mode=flow_mode)
        kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
        regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"

        row = {
            "regime_id": regime_id, "system": vals[0], "task": vals[1],
            "forcing_mode": vals[2], "k": float(vals[3]), "flow_mode": vals[4],
        }
        row.update(stats_row)
        rows.append(row)
        subsets[regime_id] = sub.copy()
        stats_map[regime_id] = stats
        names_map[regime_id] = names

    coef_df = pd.DataFrame(rows).reset_index(drop=True)
    coef_cols = [c for c in coef_df.columns if c.startswith("coef_")]

    # FIX: fill missing coefficients (term absent in regime) with 0
    if len(coef_cols) > 0:
        coef_df[coef_cols] = coef_df[coef_cols].fillna(0.0)

    return coef_df, coef_cols, subsets, stats_map, names_map

    coef_df = pd.DataFrame(rows).reset_index(drop=True)
    coef_cols = [c for c in coef_df.columns if c.startswith("coef_")]
    return coef_df, coef_cols, subsets, stats_map, names_map

def build_symbolic_features(df_in, columns=None, reduced_terms=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)
    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2
    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")
    X_sf = pd.get_dummies(sf, prefix="sf")
    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(df_in["k"].astype(float).to_numpy().reshape(-1, 1))
    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)
    if reduced_terms is not None:
        X = X.reindex(columns=reduced_terms, fill_value=0.0)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X.astype(float)

def term_stability_table(coef_df, coef_cols, n_splits=8, threshold=1e-4):
    n = len(coef_df)
    splitter = LeaveOneOut() if n <= 12 else KFold(n_splits=min(n_splits, n), shuffle=True, random_state=42)
    all_features = build_symbolic_features(coef_df).columns.tolist()
    stability = {coef: {feat: 0 for feat in all_features} for coef in coef_cols}
    fold_count = 0

    for train_idx, _ in splitter.split(coef_df):
        train_df = coef_df.iloc[train_idx].reset_index(drop=True)
        X_train = build_symbolic_features(train_df, columns=all_features)

        for coef in coef_cols:
            y = train_df[coef].to_numpy(dtype=float)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_train)
            model = LassoCV(cv=min(5, len(train_df)), random_state=fold_count + 1, max_iter=30000)
            model.fit(Xs, y)
            for feat, val in zip(all_features, model.coef_):
                if abs(val) > threshold:
                    stability[coef][feat] += 1
        fold_count += 1

    return pd.DataFrame([
        {"coefficient": coef, "term": feat, "frequency": stability[coef][feat] / max(fold_count, 1), "count": stability[coef][feat], "folds": fold_count}
        for coef in coef_cols for feat in all_features
    ])

def stable_terms_by_coefficient(coef_df, coef_cols, threshold=0.5):
    stability_df = term_stability_table(coef_df, coef_cols)
    terms_by_coef = {}
    for coef in coef_cols:
        sub = stability_df[(stability_df["coefficient"] == coef) & (stability_df["frequency"] >= threshold)]
        terms_by_coef[coef] = sub["term"].tolist()
    return terms_by_coef, stability_df

def predict_pruned_coefficients(train_df, test_df, coef_cols, terms_by_coef):
    Yhat = np.zeros((len(test_df), len(coef_cols)), dtype=float)
    for j, coef in enumerate(coef_cols):
        terms = terms_by_coef.get(coef, [])
        if len(terms) == 0:
            Yhat[:, j] = train_df[coef].mean()
            continue
        X_train = build_symbolic_features(train_df, reduced_terms=terms)
        X_test = build_symbolic_features(test_df, columns=X_train.columns, reduced_terms=terms)
        y_train = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
        model = Ridge(alpha=1.0)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)
    return Yhat

hard_block_masks_template = {
    "k_high": lambda x: x["k"].astype(float) == 7.0,
    "k_low": lambda x: x["k"].astype(float) == 3.0,
    "forcing::condition": lambda x: x["forcing_mode"].astype(str) == "condition_gap",
    "forcing::capacity": lambda x: x["forcing_mode"].astype(str) == "capacity_gap",
    "forcing::feature": lambda x: x["forcing_mode"].astype(str) == "feature_gap",
    "system::entropy": lambda x: x["system"].astype(str) == "entropy",
    "system::unevenness": lambda x: x["system"].astype(str) == "unevenness",
    "flow::linear": lambda x: x["flow_mode"].astype(str) == "linear",
    "flow::nonlinear": lambda x: x["flow_mode"].astype(str) == "nonlinear",
}

In [ ]:
# Evaluate basis class

def evaluate_basis(basis_name):
    coef_df, coef_cols, subsets, stats_map, names_map = build_coef_table(basis_name)
    terms_by_coef, stability_df = stable_terms_by_coefficient(coef_df, coef_cols, threshold=0.5)

    rows = []
    for block_name, mask_fn in hard_block_masks_template.items():
        test_mask = mask_fn(coef_df)
        train_coef = coef_df.loc[~test_mask].reset_index(drop=True)
        test_coef = coef_df.loc[test_mask].reset_index(drop=True)
        if len(test_coef) == 0 or len(train_coef) < 5:
            continue

        beta_hat_mat = predict_pruned_coefficients(train_coef, test_coef, coef_cols, terms_by_coef)

        for i, rid in enumerate(test_coef["regime_id"].tolist()):
            sub = subsets[rid]
            flow_mode = test_coef.loc[i, "flow_mode"]
            cols = [c.replace("coef_", "") for c in coef_cols]
            beta_true = test_coef.loc[i, coef_cols].to_numpy(dtype=float)
            beta_hat = beta_hat_mat[i]
            y_true = sub["predicted_flow"].to_numpy(dtype=float)
            y_pred = predict_with_beta(sub, basis_name, beta_hat, stats=stats_map[rid], flow_mode=flow_mode, columns=cols)
            rows.append({
                "basis": basis_name, "block": block_name, "regime_id": rid,
                "flow_mode": flow_mode, "forcing_mode": test_coef.loc[i, "forcing_mode"],
                "system": test_coef.loc[i, "system"], "task": test_coef.loc[i, "task"], "k": test_coef.loc[i, "k"],
                "n_field_terms": len(cols),
                "n_stable_chart_terms": sum(len(v) for v in terms_by_coef.values()),
                "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
                "field_rmse": math.sqrt(mean_squared_error(y_true, y_pred)),
                "traj_rmse": trajectory_gap(sub, basis_name, beta_true, beta_hat, stats_ref=stats_map[rid], stats_cmp=stats_map[rid], flow_mode=flow_mode, columns=cols),
            })

    return pd.DataFrame(rows), coef_df, stability_df, terms_by_coef

basis_names = ["raw_6", "structured_interaction", "compact_interaction", "block_conditioned"]

eval_frames = []
basis_details = {}
for b in basis_names:
    print("Evaluating", b)
    ev, cdf, st, tbc = evaluate_basis(b)
    eval_frames.append(ev)
    basis_details[b] = {"coef_df": cdf, "stability_df": st, "terms_by_coef": tbc}

all_eval_df = pd.concat(eval_frames, ignore_index=True)
all_eval_df.to_csv(os.path.join(OUTPUT_DIR, "all_basis_eval_raw.csv"), index=False)

summary_df = all_eval_df.groupby("basis")[["coef_rmse", "field_rmse", "traj_rmse", "n_field_terms", "n_stable_chart_terms"]].mean().reset_index().sort_values("field_rmse")
display(summary_df)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "basis_class_summary.csv"), index=False)

In [ ]:
# Universality score

def universality_scores(eval_df, metric="traj_rmse", tolerance=0.05):
    block_summary = eval_df.groupby(["block", "basis"])[metric].mean().reset_index()
    rows = []
    for block, sub in block_summary.groupby("block"):
        best = sub[metric].min()
        for _, row in sub.iterrows():
            rows.append({
                "block": block, "basis": row["basis"], "metric": metric,
                "value": row[metric], "best": best,
                "within_tolerance": bool(row[metric] <= (1 + tolerance) * best),
                "is_best": bool(np.isclose(row[metric], best) or row[metric] == best),
            })
    by_block = pd.DataFrame(rows)
    score = by_block.groupby("basis").agg(
        universality_score=("within_tolerance", "mean"),
        win_rate=("is_best", "mean"),
        mean_metric=("value", "mean"),
        n_blocks=("block", "nunique"),
    ).reset_index().sort_values(["universality_score", "win_rate", "mean_metric"], ascending=[False, False, True])
    return score, by_block

U_TOL = 0.05
u_score_df, u_block_df = universality_scores(all_eval_df, metric="traj_rmse", tolerance=U_TOL)
display(u_score_df)
u_score_df.to_csv(os.path.join(OUTPUT_DIR, "universality_score_by_basis.csv"), index=False)
u_block_df.to_csv(os.path.join(OUTPUT_DIR, "universality_by_block.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(u_score_df["basis"], u_score_df["universality_score"])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel(f"Uτ, τ={U_TOL}")
axes[0].set_title("Universality score")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(u_score_df["basis"], u_score_df["win_rate"])
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("strict win rate")
axes[1].set_title("Block win rate")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure1_universality_and_win_rate.png"), dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# Summary and block plots

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
plot_df = summary_df.copy()
for ax, metric in zip(axes, ["coef_rmse", "field_rmse", "traj_rmse", "n_stable_chart_terms"]):
    ax.bar(plot_df["basis"], plot_df[metric])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure2_basis_class_summary.png"), dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(summary_df["n_stable_chart_terms"], summary_df["traj_rmse"], s=80)
for _, row in summary_df.iterrows():
    ax.text(row["n_stable_chart_terms"] + 0.3, row["traj_rmse"], row["basis"], fontsize=9)
ax.set_xlabel("stable chart terms")
ax.set_ylabel("trajectory RMSE")
ax.set_title("Stability/complexity vs trajectory error")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure3_stability_complexity_tradeoff.png"), dpi=220, bbox_inches="tight")
plt.show()

block_summary = all_eval_df.groupby(["block", "basis"])[["field_rmse", "traj_rmse"]].mean().reset_index()
display(block_summary.sort_values(["block", "traj_rmse"]))

for metric in ["field_rmse", "traj_rmse"]:
    pivot = block_summary.pivot(index="block", columns="basis", values=metric)
    fig, ax = plt.subplots(figsize=(13, 5))
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel(metric)
    ax.set_title(f"Hard-block {metric} by basis class")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"figure4_block_{metric}_by_basis.png"), dpi=220, bbox_inches="tight")
    plt.show()

block_summary.to_csv(os.path.join(OUTPUT_DIR, "block_summary_by_basis.csv"), index=False)

In [ ]:
# Stability summary

stability_all = pd.concat([
    details["stability_df"].assign(basis=basis)
    for basis, details in basis_details.items()
], ignore_index=True)
stability_all.to_csv(os.path.join(OUTPUT_DIR, "symbolic_chart_stability_all_basis.csv"), index=False)

stable_count = stability_all[stability_all["frequency"] >= 0.5].groupby("basis").size().reset_index(name="stable_terms")
stable_mean = stability_all.groupby("basis")["frequency"].mean().reset_index(name="mean_selection_frequency")
stability_summary = stable_count.merge(stable_mean, on="basis", how="outer").fillna(0)
display(stability_summary)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(stability_summary["basis"], stability_summary["stable_terms"])
ax.set_title("Stable symbolic chart terms by basis")
ax.set_ylabel("count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure5_stable_terms_by_basis.png"), dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# Representative reconstruction

paired = all_eval_df.pivot_table(index=["block", "regime_id"], columns="basis", values="field_rmse").dropna()
if "raw_6" in paired.columns and "structured_interaction" in paired.columns:
    paired["structured_gain"] = paired["raw_6"] - paired["structured_interaction"]
    rep_block, rep_rid = paired.sort_values("structured_gain", ascending=False).index[0]
else:
    rep_block, rep_rid = all_eval_df.iloc[0][["block", "regime_id"]]

print("Representative:", rep_block, rep_rid)

fig, axes = plt.subplots(1, len(basis_names) + 1, figsize=(4 * (len(basis_names)+1), 4), sharey=True)
sub_ref = None
y_true_ref = None

for j, panel in enumerate(["empirical"] + basis_names):
    ax = axes[j]
    if panel == "empirical":
        coef_df_b = basis_details["raw_6"]["coef_df"]
        subsets = build_coef_table("raw_6")[2]
        sub = subsets[rep_rid]
        vals = sub["predicted_flow"].to_numpy(dtype=float)
        sub_ref = sub
        y_true_ref = vals
    else:
        coef_df_b = basis_details[panel]["coef_df"]
        coef_cols = [c for c in coef_df_b.columns if c.startswith("coef_")]
        terms_by_coef = basis_details[panel]["terms_by_coef"]
        test_mask = hard_block_masks_template[rep_block](coef_df_b)
        train_coef = coef_df_b.loc[~test_mask].reset_index(drop=True)
        test_coef = coef_df_b.loc[test_mask].reset_index(drop=True)
        test_row = test_coef[test_coef["regime_id"] == rep_rid].reset_index(drop=True)
        sub = sub_ref
        if len(test_row) == 0:
            vals = np.full_like(y_true_ref, np.nan)
        else:
            beta_hat = predict_pruned_coefficients(train_coef, test_row, coef_cols, terms_by_coef)[0]
            stats = basis_stats(sub)
            flow_mode = test_row.loc[0, "flow_mode"]
            cols = [c.replace("coef_", "") for c in coef_cols]
            vals = predict_with_beta(sub, panel, beta_hat, stats=stats, flow_mode=flow_mode, columns=cols)

    sc = ax.scatter(sub["condition_coord"], sub["residual"], c=vals, s=16)
    ax.set_title(panel)
    ax.set_xlabel("condition c")
    if j == 0:
        ax.set_ylabel("residual r")
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f"Representative reconstruction: {rep_rid}", y=1.03)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure6_representative_reconstruction.png"), dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# Final verdict

best_field = summary_df.sort_values("field_rmse").iloc[0]
best_traj = summary_df.sort_values("traj_rmse").iloc[0]
best_u = u_score_df.sort_values(["universality_score", "win_rate", "mean_metric"], ascending=[False, False, True]).iloc[0]

raw = summary_df[summary_df["basis"] == "raw_6"].iloc[0]
structured = summary_df[summary_df["basis"] == "structured_interaction"].iloc[0]

raw_u = u_score_df[u_score_df["basis"] == "raw_6"]["universality_score"].iloc[0]
structured_delta = float(structured["field_rmse"] - raw["field_rmse"])

if best_field["basis"] == "raw_6" and best_u["basis"] == "raw_6":
    final_verdict = "raw_6 remains final paper-facing basis"
elif best_u["basis"] != "raw_6" and best_u["universality_score"] > raw_u:
    final_verdict = f"{best_u['basis']} improves universality; consider as final or appendix challenger"
elif best_field["basis"] != "raw_6" and best_field["field_rmse"] <= raw["field_rmse"] * 0.98:
    final_verdict = f"{best_field['basis']} improves field RMSE materially; inspect as possible final basis"
else:
    final_verdict = "raw_6 remains final; structured bases are validation/appendix comparisons"

verdict_df = pd.DataFrame([{
    "best_field_basis": best_field["basis"],
    "best_field_rmse": float(best_field["field_rmse"]),
    "best_traj_basis": best_traj["basis"],
    "best_traj_rmse": float(best_traj["traj_rmse"]),
    "best_universality_basis": best_u["basis"],
    "best_universality_score": float(best_u["universality_score"]),
    "raw_field_rmse": float(raw["field_rmse"]),
    "raw_traj_rmse": float(raw["traj_rmse"]),
    "structured_minus_raw_field_rmse": structured_delta,
    "final_verdict": final_verdict,
}])
display(verdict_df)
verdict_df.to_csv(os.path.join(OUTPUT_DIR, "final_basis_class_verdict.csv"), index=False)

paragraph = f'''
## Structured interaction basis test

Notebook 70 tested whether the final field should replace the raw six-term basis with an interaction-aware or block-conditioned basis. The best field basis was `{best_field["basis"]}` with field RMSE {best_field["field_rmse"]:.4f}; the best trajectory basis was `{best_traj["basis"]}` with trajectory RMSE {best_traj["traj_rmse"]:.4f}; and the best universality basis was `{best_u["basis"]}` with Uτ={best_u["universality_score"]:.3f}. The final verdict is: `{final_verdict}`. This result separates physical alignment from coordinate independence: full orthogonalization was rejected in Notebook 69, while structured interaction bases provide a controlled challenger that preserves coupling rather than forcing independence.
'''
with open(os.path.join(OUTPUT_DIR, "structured_interaction_basis_paragraph.md"), "w", encoding="utf-8") as f:
    f.write(paragraph.strip() + "\n")
display(Markdown(paragraph))

In [ ]:
manifest = {
    "data_source": data_source,
    "synthetic_fallback": bool(USE_SYNTHETIC_FALLBACK),
    "basis_names": basis_names,
    "universality_tolerance": U_TOL,
    "final_verdict": final_verdict,
    "outputs": sorted(os.listdir(OUTPUT_DIR)),
}

with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

display(pd.DataFrame({"output_file": sorted(os.listdir(OUTPUT_DIR))}))
print("Saved outputs under:", OUTPUT_DIR)

## Final statement

Notebook 70 tests the correct refinement after Notebook 69:

> Do not orthogonalize away interaction. Preserve coupling and test whether structured interaction coordinates outperform raw coordinates.

Use the final verdict to decide whether the paper-facing equation remains `raw_6` or adopts a structured interaction basis.